# Module 4: Cross-ecosystem comparison (all bulk)

**Biological question.** When methods calibration passes, do independently generated lung contrasts agree at the pathway level -- and what claims stay unlicensed when they do not?

**Learning objectives.** On completing this module you will be able to:
- Separate method validation from biology concordance (Analyze, Evaluate)
- Read shared-mechanism overlap against chance, not global NES Spearman alone (Analyze)
- Recognize when a contrast is uninterpretable for a directional question even after calibration passes (Evaluate)
- Communicate an interpretation that states what the analysis does not show (Evaluate)

**Bloom.** Analyze, Evaluate

**Prerequisites.** Modules 1 to 3 complete; conda env `CFDE_lung_env` (or the pinned pip venv); Module 2 integrated object and pseudobulk tables present.

**Time.** Instructional: ~30 min. Compute (measured from `module4_run_params.json`, sensitivity gated off): notebook wall ~5 min / ~301 s compute; peak RSS ~5.5 GB. Course RAM: 16 GB recommended, 8 GB minimum (Module 2 DE peaks at ~7.6 GB). Free disk: several GB for tables/figures.

**Data required.**
| Ecosystem | Resource | Role | Notes |
|---|---|---|---|
| NIH (not CFDE) | GSE150910 | `H_DISEASE`: IPF vs control | External disease arm |
| CFDE | GTEx v10 lung | `H_AGING_GTEX`: older vs younger (gene_reads); TPM for levels/bridge | CFDE bulk |
| NASA | OSD-248 | `M_FLIGHT`: ISS-T flight vs duration-matched GC | Mouse bulk |
| CFDE | HuBMAP | `H_AGING_HUBMAP`: older vs younger donors (2 vs 2 illustration); level bridge | snRNA-derived |

No CFDE program supplies fibrotic human lung, which is why Module 4 goes external (GEO GSE150910).

**Spine.** Four resources, four within-resource contrasts, plus a HuBMAP-GTEx level bridge. Native levels are shown side-by-side on purpose. Contrasts (logFC / NES) are what travel. Counts are never merged across species or studies.

**Honesty / close (read after the analysis cells).**

1. **Methods pass:** ERCC ExFold OLS slope ~0.989, R2 ~0.882; GeneLab ISS-T Spearman rho ~0.862; HuBMAP-GTEx bridge composition-weighted rho ~0.74 (max-across ~0.64).
2. **Global NES is supporting only.** Hallmark+Reactome Spearman is near-null to negative across biology pairs (disease-aging ~-0.197). These contrasts are not broadly equivalent.
3. **Disease-aging (not above chance):** **five separate concordant pathways**, grouped only by direction. Down: cholesterol biosynthesis, SREBF activation. Up: EMT, ECM organization, keratinization. Leading-edge clustering at Jaccard 0.5 merged none of them, so five is the reported granularity. 5 observed against a permutation null mean of 3.0, p ~0.169: it does not survive as a shared-mechanism claim.
4. **ECM correction:** a negative global coefficient can suggest ECM runs opposite between IPF and aging; the pathway screen shows Extracellular Matrix Organization **up in both**. That is why the coefficient is not the headline.
5. **Flight pairs are uninterpretable, not null,** while pathway passers are unanimously down (`p_up = 0`). ERCC-in-size-factors tested and rejected (ground ERCC fraction higher; exclusion leaves `p_up = 0`). Adult globin library share ~1.33% flight vs ~0.96% ground -- not material. Close the directional skew as a stated open question (circadian / carcass / other), not a calibration failure.
6. **Aging-flight:** **six concordant pathways, five after clustering** merged the two Respiratory Electron Transport terms. Oxidative phosphorylation, fatty-acid beta-oxidation and mitochondrial protein import stay distinct. The six together are the only set above permutation, 6 against 2.48, p ~0.026. **Myc Targets V2 is adjacent and reported separately.** Three of the six are suspect-flagged, so do not firm it up.
7. **Three-way intersection is empty.** All concordant pathways stay **tier 3** (Jaccard ~0.21 to 0.48; no natural tier 1/2 cut on this run).

**Run order.** Run the analysis sections in order; the notebook carries state between cells.


## How this module fits the course

Modules 1 to 3 stay inside HuBMAP. This module puts four independently generated resources beside
each other, HuBMAP, GTEx, GEO and NASA OSDR, and asks the only question that survives the differences
between them: do their contrasts agree, rather than do their values agree.

The notebook builds one contrast within each resource, calibrates the flight arm against its
spike-ins, maps orthologs, then screens for pathways moving the same way in more than one resource
against a permutation null.

Levels never travel between these resources. Units, tissue handling and library chemistry all differ,
so a side-by-side table of expression values is shown once to demonstrate why it cannot be
interpreted, and every claim after that is made at the level of contrasts.


In [ ]:
from pathlib import Path
import os, sys, time
MODULE_ROOT = Path.cwd().resolve()
if MODULE_ROOT.name == "notebooks":
    MODULE_ROOT = MODULE_ROOT.parent
if not (MODULE_ROOT / "scripts").is_dir():
    MODULE_ROOT = next(
        (p for p in MODULE_ROOT.parents if (p / "scripts").is_dir()), MODULE_ROOT
    )
os.chdir(MODULE_ROOT)
sys.path.insert(0, str(MODULE_ROOT))

from scripts.common.paths import ensure_output_dirs, load_config, resolve
from scripts.common.runtime import finalize_timing, peak_rss_mb
from scripts.nb06_crossres.compare import (
    compare_configured_nes_pairs,
    hubmap_gtex_bridge_table,
    is_ercc_symbol,
    nes_sensitivity_table,
    plot_nes_scatter,
    plot_nes_sensitivity,
    plot_pseudobulk_vs_gtex,
    plot_volcano,
    prerank_contrast,
    resolve_nes_sensitivity,
    sensitivity_enabled,
)
from scripts.nb06_crossres.contrasts import (
    contrast_h_aging_gtex,
    contrast_h_aging_hubmap,
    contrast_h_disease,
    contrast_m_flight,
    gtex_age_bracket_inventory,
    logfc_series,
)
from scripts.nb06_crossres.ercc_calib import (
    ercc_calibration_frame,
    ercc_calibration_summary,
    plot_ercc_calibration,
)
from scripts.nb06_crossres.export import save_module4_outputs
from scripts.nb06_crossres.load import native_levels_panel, sample_counts_table
from scripts.nb06_crossres.orthologs import load_ortholog_table, ortholog_loss_record
import pandas as pd
from scipy.stats import spearmanr

cfg = load_config()
ensure_output_dirs(cfg)
fig_dir = resolve(cfg, "outputs_figures")
panel = list(cfg["module4"]["params"]["gene_panel"])
_m4_timing = {
    "_t0": time.perf_counter(),
    "_compute_seconds": None,
    "peak_rss_mb": None,
    "peak_rss_mb_start": peak_rss_mb(),
    "peak_rss_note": (
        "process peak RSS so far (ru_maxrss); monotone across a loop, not per-step"
    ),
}
print("GTEx age brackets (declare before DE):")
gtex_inv = gtex_age_bracket_inventory(cfg)
display(gtex_inv)
display(sample_counts_table(cfg))


## 1. Native expression levels across resources

TPM vs linear pseudobulk vs raw counts; human vs mouse. Display the panel before deciding what travels.


In [ ]:
native = native_levels_panel(cfg)
display(native)


**Finding.** Native levels are not comparable across these resources. The side-by-side panel is the lesson: contrasts (logFC / NES), not raw levels, are what can travel. See `outputs/tables/module4_native_levels_panel.tsv`.


## 2. Four within-resource contrasts and calibration

- **H_DISEASE**: `~ condition` only (published analysis used covariates).
- **H_AGING_GTEX**: gene_reads; predeclared brackets 20-39 vs 60-79; one sample per subject.
- **M_FLIGHT**: ISS-T flight vs GC (renamed from M_UNLOAD; these are flight samples, not hindlimb unloading).
- **H_AGING_HUBMAP**: Donor_3+7 vs Donor_1+3 (2 vs 2 **illustration only**). Do not loosen padj.

**Polarity.** Reference levels are set explicitly (`Treatment(reference)` plus an explicit
DeseqStats contrast). Alphabetical factor order would have inverted three of the four contrasts
(`IPF` sorts before `control`; `older` before `younger`).

**ERCC.** Flight=Mix1, GC=Mix2. Spike-ins calibrate and cannot normalize this contrast.
`ERCC-` rows stay in the DE table for ExFold calibration; size factors are estimated on
endogenous genes only. Measured ERCC read fraction is slightly *higher* in ground than flight,
and excluding spike-ins from size factors does **not** move pathway `p_up` off zero. The
all-down pathway-passer pattern is therefore not that size-factor artifact.


In [ ]:
de_h = contrast_h_disease(cfg)
de_age_gtex = contrast_h_aging_gtex(cfg)
de_m = contrast_m_flight(cfg, drop_low_qa=False)
de_m_sens = contrast_m_flight(cfg, drop_low_qa=True)
de_age_hub = contrast_h_aging_hubmap(cfg)

display(de_h.sort_values("padj").head(5))
display(de_age_gtex.sort_values("padj").head(5))
display(de_m.sort_values("padj").head(5))
print("H_AGING_HUBMAP padj<0.05:", int((de_age_hub["padj"] < 0.05).sum()), "(illustration; expect few/none)")
display(de_age_hub.sort_values("padj").head(5))

de_m_no_ercc = de_m[~de_m["gene_symbol"].map(is_ercc_symbol)]
print("Top M_FLIGHT genes excluding ERCC- spike-ins:")
display(de_m_no_ercc.sort_values("padj").head(10))

ercc_cal = ercc_calibration_frame(de_m, cfg)
ercc_stats = ercc_calibration_summary(ercc_cal)
print("ERCC ExFold calibration:", ercc_stats)
plot_ercc_calibration(ercc_cal, fig_dir / "module4_ercc_exfold_calibration.png", summary=ercc_stats)

subset_rel = (cfg.get("module4", {}).get("data") or {}).get(
    "osd248_genelab_isst_de_subset",
    "outputs/tables/module4_osd248_genelab_isst_de_subset.tsv",
)
subset_path = MODULE_ROOT / subset_rel
genelab_subset = pd.read_csv(subset_path, sep="\t") if subset_path.exists() else pd.DataFrame()
print("GeneLab ISS-T DE subset:", subset_path, "rows", len(genelab_subset))

for de, fname, title in [
    (de_h, "module4_volcano_h_disease.png", "H_DISEASE: IPF vs control"),
    (de_age_gtex, "module4_volcano_h_aging_gtex.png", "H_AGING_GTEX: older vs younger"),
    (de_m, "module4_volcano_m_flight.png", "M_FLIGHT: flight vs GC"),
    (de_age_hub, "module4_volcano_h_aging_hubmap.png", "H_AGING_HUBMAP: 2 vs 2 illustration"),
]:
    plot_volcano(de, fig_dir / fname, title=title, panel=panel)


## 3. Ortholog mapping and pathway concordance

Global NES Spearman (Hallmark + Reactome) is a **supporting** line: these contrasts are not broadly
equivalent. It can also mislead -- Extracellular Matrix Organization passes **up in both** IPF and
GTEx aging even when disease-aging rho is negative.

The **headline** is shared-mechanism concordance: pathways that pass FDR q<0.25 and |NES|>=1.5 in each
contrast with agreeing sign, scored against analytic and permutation nulls. Three biology pairs plus
the three-way intersection. HuBMAP aging stays a footnote.

All concordant pathways ship as **tier 3** (Jaccard ~0.21 to 0.48; no natural tier 1/2 cut). Suspect
ribosome / translation / OxPhos terms are flagged, never excluded.

**How to read the four cells.** Disease-aging: five separate pathways, not above chance.
Disease-flight and three-way: **uninterpretable** while flight pathway passers are all down.
Aging-flight: six pathways in five clusters above permutation, Myc Targets V2 adjacent; three suspect-flagged.


In [ ]:
ortho = load_ortholog_table(cfg)
loss = ortholog_loss_record(de_h["gene_symbol"], de_m["human_symbol"].dropna(), ortho)
display(loss)

m_gsea = de_m.dropna(subset=["human_symbol"]).copy()
m_gsea["gene_symbol"] = m_gsea["human_symbol"].astype(str).str.upper()
nes_by_id = {
    "H_DISEASE": prerank_contrast(logfc_series(de_h), cfg, contrast_id="H_DISEASE"),
    "H_AGING_GTEX": prerank_contrast(logfc_series(de_age_gtex), cfg, contrast_id="H_AGING_GTEX"),
    "M_FLIGHT": prerank_contrast(logfc_series(m_gsea), cfg, contrast_id="M_FLIGHT"),
    "H_AGING_HUBMAP": prerank_contrast(logfc_series(de_age_hub), cfg, contrast_id="H_AGING_HUBMAP"),
}
nes_pair_summary, nes_pair_details = compare_configured_nes_pairs(nes_by_id, cfg)
display(nes_pair_summary)

headline = list(cfg["module4"]["params"].get("nes_headline_pair") or ["H_DISEASE", "H_AGING_GTEX"])
headline_key = f"{headline[0]}__{headline[1]}"
nes_joined, nes_stats = nes_pair_details[headline_key]
print("Headline pair:", nes_stats)
for key, (joined, stats) in nes_pair_details.items():
    safe = key.replace("__", "_vs_")
    plot_nes_scatter(
        joined,
        fig_dir / f"module4_nes_scatter_{safe}.png",
        stats=stats,
        highlight=cfg["module4"]["params"].get("highlight_pathways"),
        pair_label=stats.get("pair_label"),
    )

# Sensitivity table: shipped default loads committed TSV (module4.sensitivity.enabled=false).
# Set enabled=true to re-prerank Hallmark+Reactome+GO BP (high RAM; reproduces the asset).
if sensitivity_enabled(cfg):
    print("sensitivity.enabled=true: re-preranking with GO BP")
    nes_sens, sens_meta = resolve_nes_sensitivity(
        cfg,
        logfc_x=logfc_series(de_h),
        logfc_y=logfc_series(de_age_gtex),
        contrast_x="H_DISEASE",
        contrast_y="H_AGING_GTEX",
    )
else:
    print("sensitivity.enabled=false: loading committed module4_nes_sensitivity.tsv")
    nes_sens, sens_meta = resolve_nes_sensitivity(cfg)
print(sens_meta)
display(nes_sens)
plot_nes_sensitivity(nes_sens, fig_dir / "module4_nes_sensitivity.png")


## 4. HuBMAP to GTEx level bridge

Never z-score across units. Log1p once. Compare composition-weighted vs max-across vectors against GTEx mean TPM, then read which tracks more closely.


In [ ]:
bridge = hubmap_gtex_bridge_table(cfg)
plot_pseudobulk_vs_gtex(bridge, fig_dir / "module4_pseudobulk_vs_gtex.png", panel=panel)

finalize_timing(_m4_timing)
fig_paths = {k: fig_dir / f"module4_{k}.png" for k in [
    "volcano_h_disease", "volcano_h_aging_gtex", "volcano_m_flight", "volcano_h_aging_hubmap",
    "nes_scatter", "nes_sensitivity", "pseudobulk_vs_gtex",
]}
fig_paths["ercc_calibration"] = fig_dir / "module4_ercc_exfold_calibration.png"
# Register every NES pair scatter the analysis cell wrote (notebook/smoke parity).
for _key in list(nes_pair_details.keys()):
    _safe = _key.replace("__", "_vs_")
    _p = fig_dir / f"module4_nes_scatter_{_safe}.png"
    if _p.exists():
        fig_paths[f"nes_scatter_{_safe}"] = _p

out = save_module4_outputs(
    cfg,
    sample_counts=sample_counts_table(cfg),
    native_levels=native,
    contrast_h=de_h,
    contrast_m=de_m,
    contrast_m_sens=de_m_sens,
    contrast_aging_gtex=de_age_gtex,
    contrast_aging_hubmap=de_age_hub,
    ortholog_loss=loss,
    nes_by_id=nes_by_id,
    nes_pair_summary=nes_pair_summary,
    nes_pair_details=nes_pair_details,
    bridge_df=bridge,
    figure_paths={k: v for k, v in fig_paths.items() if Path(v).exists()},
    nes_sensitivity=nes_sens,
    ercc_calibration=ercc_cal,
    genelab_subset=genelab_subset if not genelab_subset.empty else None,
    gtex_bracket_inventory=gtex_inv,
    extras={
        **{k: v for k, v in _m4_timing.items() if not str(k).startswith("_t")},
        "ercc_calibration": ercc_stats,
        "genelab_isst_de_subset": str(subset_path),
        "n_genelab_subset_rows": int(len(genelab_subset)),
    },
)
print(
    "Wrote",
    out["methods"],
    f"_compute_seconds={_m4_timing.get('_compute_seconds')}",
    f"peak_rss_mb={_m4_timing.get('peak_rss_mb')}",
)
print("Done.")


from scripts.nb06_crossres.ercc_calib import ercc_read_fraction_by_sample, ercc_read_fraction_arm_summary

ercc_frac = ercc_read_fraction_by_sample(cfg)
ercc_frac_path = resolve(cfg, "outputs_tables") / "module4_ercc_read_fraction_by_sample.tsv"
ercc_frac.to_csv(ercc_frac_path, sep="\t", index=False)
print("ERCC fraction by arm:", ercc_read_fraction_arm_summary(ercc_frac))
display(ercc_frac.groupby("condition")["ercc_fraction"].describe())

from scripts.nb06_crossres.export import append_mechanism_section
from scripts.nb06_crossres.mechanisms import run_mechanism_analysis, save_mechanism_outputs
mech = run_mechanism_analysis(cfg)
save_mechanism_outputs(cfg, mech)
append_mechanism_section(
    out["methods"],
    null_df=mech["concordance_null"],
    summary=mech["mechanism_summary"],
    mechanisms=mech["mechanisms"],
)


**Finding.** Composition-weighted HuBMAP pseudobulk tracks GTEx bulk more closely than max-across (rho ~0.74 vs ~0.64). This bridge uses the composition-weighted vector, not the donor-summed counts used for `H_AGING_HUBMAP`. See `outputs/tables/module4_hubmap_gtex_bridge.tsv` and `outputs/figures/module4_pseudobulk_vs_gtex.png`.


## Questions this result raises

1. If methods calibration succeeds and biology concordance stays weak, what claim is still licensed?
2. Why is global NES Spearman the wrong headline for shared mechanisms?
3. Disease-aging gives five separate pathways, not above chance. What would
   promote that from hypothesis to claim?
4. Flight pathway passers are all down after ERCC and globin checks failed to explain it. What
   confound remains on the table?
5. Why keep flagged OxPhos / RET terms in the aging-flight set instead of dropping them?
6. What does an empty three-way intersection mean for "spaceflight as accelerated aging" rhetoric?
7. HuBMAP aging has zero genes at padj < 0.05. How should a learner read its NES passers?
8. When should `module4.sensitivity.enabled` be turned on, and what does it cost?
